# மனிதர்-இன்-தி-லூப்: முன்-செயல் கவடுகள், அபாய தகுதிச் சான்று மற்றும் நிபந்தனை பதிவேடு

இந்த பாடக்குறிப்பின் README மனிதர்-இன்-தி-லூப்பை அறிமுகப்படுத்துகிறது, இதில் முகவர் பதில் தயாரித்த பின் பயனரிடம் `அனுமதி` அல்லது `மறுப்பு` கேட்கும் ஒரு சிறிய துண்டின் மூலம். அந்த மாதிரியும் ஒரு நல்ல துவக்கம், ஆனால் உற்பத்தி HITL அமல்படுத்தல்கள் பொதுவாக மூன்று கூடுதல் கூறுகளை தேவைப்படுத்துகின்றன:

1. முகவரவர் ஒரு அபாயமிக்க படியை செயல்படுத்துவதற்கு **முன்பு** இயங்கும் **முன்-செயல் கவடு**, ஆகையால் செலவு, மாற்ற முடியாத தன்மை மற்றும் தாமதம் கட்டுப்பாட்டில் வைக்கப்படலாம்.
2. **அபாய தகுதிச் சான்று**, அதாவது குறைந்த அபாய செயல்கள் தானாகவே செயல்படுத்தப்படுகின்றன, நடுத்தர அபாய செயல்கள் தொகுப்பு அனுமதி பெறுகின்றன, மற்றும் உயர் அபாய செயல்கள் மட்டுமே மனிதரால் தடுக்கப்படுகின்றன.
3. ஒரு **ஆடிட் பதிவு மற்றும் திருத்த முறை**, அதன்படி ஒவ்வொரு கவடு முடிவு JSONL வடிவில் பதிவு செய்யப்படுகின்றது, மறுப்பு ஏற்பட்டால் முகவருக்கு கட்டமைக்கப்பட்ட காரணத்துடன் மீண்டும் கேட்டு, வெறும் `மீட்டமைத்தல்...` என்று அச்சிடுவதைக் காட்டாது.

இந்த நோட்டுபுக் இவை அனைத்தையும் `06-system-message-framework.ipynb` உள்ள அதே அடிப்படைகளின் மேல் கட்டமைக்கிறது. இது முழுமையாக `DEMO_MODE = True` (எந்தவொரு நேரடி உள்ளீடு வேண்டாம்) அல்லது உண்மையான `input()` கேள்விகளுடன் `DEMO_MODE = False` இல் இயங்கும். குறிப்பு: DEMO_MODE இல் மூன்றாவது இலக்கின் மீண்டும் முயற்சி செயல் குறிப்பு நிரலாக உள்ளது எனவே முழு செயல்முறை தெளிவாக தெரிகிறது. உண்மையான திருத்தம் சார்ந்த மறு வகைப்படுத்தல் `DEMO_MODE = False` மற்றும் ஒரு இயக்குநர் தேவைப்படுகின்றன.

**வரம்பு வெளியே (பிற பாடங்களில் கையாளப்படும்):** அங்கீகாரம் மற்றும் அணுகல் கட்டுப்பாடு (பாடம் 06 README அச்சுறுத்தல் 2), கருவி-கால் நடுநிலை மென்பொருள் (பாடம் 14 MAF விரிவான ஆய்வு), பன்முகவர் விவாத நுட்பங்கள்.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## ஒழுங்கு 1: முன்-செயல் கதவு

README-யின் HITL துணுக்கம் முதலில் முகவர்(agent)-ஐ அழைத்து, பிறகு பயனரிடம் வெளியீட்டை அங்கீகாரம் செய்ய கேட்கிறது. அது ஒரு **பிறகு-செயல்** ஓட்டமாகும். முகவர் ஏற்கனவே செயல்பட்டுவிட்டார், ஆகவே LLM அழைப்புக் கட்டணம் ஏற்கனவே செலுத்தப்பட்டு இருக்கிறது, மற்றும் ஏதேனும் பக்கவிளைவு (மின்னஞ்சல் அனுப்பு, தரவுத்தள வரி எழுதுதல், கருத்து பதிவு செய்தல்) ஏற்கனவே நடந்துவிட்டது.

ஒரு **முன்-செயல்** ஓட்டம், முகவர் அபாயகரமான படியை செயல்படுத்துவதற்கு முன் கதவை நுழைக்கிறது. முகவர் செயலை முன்மொழிகிறார், கதவு அதை செயல்படுத்துமா என முடிவு செய்கிறது, மற்றும் அங்கீகாரம் கிடைத்த பிறகு மட்டுமே பக்கவிளைவு ஏற்படுகிறது.

| அம்சம் | பிறகு-செயல் அங்கீகாரம் (README துணுக்கம்) | முன்-செயல் கதவு (இந்த நோட்புக்) |
|---|---|---|
| அங்கீகாரம் எப்போது இயங்குமா? | முகவர் வெளியீடு தயாரித்த பிறகு | எந்த பக்கவிளைவும் செயல்பட முன்னர் |
| மறுத்தல் மீது LLM செலவு | ஏற்கனவே செலுத்தப்பட்டு விட்டது | செயலுக்கு அல்ல, முன்மொழிவுக்கு மட்டுமே செலுத்தப்படுகிறது |
| திருப்ப முடியாத பக்கவிளைவுகள் | சாத்தியமானவை (செயல் ஏற்கனவே நடந்துவிட்டது) | தடைக்கப்பட்டது |
| ஆய்வுப் புகுத்தல் தெளிவுத்தன்மை | அங்கீகாரம் அச்சிடும் கூற்றாகும் | அங்கீகாரம் கால அட்டவணையுடன் கூடிய JSON பதிவாகும், செயல், காரணம் |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## மாதிரி 2: ஆபத்து நிலை வகை

ஒவ்வொரு செயலுக்கும் மனித அனுமதி அவசியம் இல்லை. ஒரு பொது API மீது மட்டும் படிக்க கூடிய தேடல், ஒரு வாடிக்கையாளர் மின்னஞ்சலை அனுப்புவதைவிட வேறுபட்ட ஆபத்துகள் கொண்டவை. இரண்டும் ஒரே மாதிரியாக நடத்துவது ஆபரேட்டரின் கவனத்தை வீணாக்கி, முகவரின் வேகத்தை குறைக்கிறது.

ஒரு எளிய 3-நிலை மாதிரி:

| நிலை | உதாரணங்கள் | அனுமதி ஓட்டம் |
|---|---|---|
| `குறைவானது` (படிக்க மட்டும்) | அறிவுத் தளத்தில் தேடல், விமான விருப்பங்களை பார்க்க, ஒரு பொது வலைப்பக்கம் பெறுதல் | தானாகச் செயல்படுத்தவும், ஆய்வுக்காக பதிவு செய்யவும் |
| `முதன்மை` (சிறிய மாற்றம்) | முடிவை தளர்த்துதல், கணக்கை அதிகரித்தல், நினைவூட்டலை நிர்ணயித்தல் | தானாகச் செயல்படுத்தவும், ஆனால் தினசரி மதிப்பாய்வுக்கு தொகுக்கப்படுகிறது |
| `உயர்நிலை` (வெளிப்புறம் எதிர்கொள்ளும் அல்லது மாற்றமுடியாத) | மின்னஞ்சல் அனுப்புதல், கார்டுக்கு கட்டணம் வசூலித்தல், பொது சேனலில் பதிவு செய்வது | மனித அனுமதிக்கு தடை வைக்கப்படும் |

இது ஒரு வகை தீர்வு மட்டுமே. உற்பத்தி முறைகள் மிகச் சிறிய நிலைகள் கொண்டிருக்கலாம் (உதாரணமாக, AWS IAM அனுமதி நிலைகள், வேட இடைநிலை அணுகல்நிலைகள்). கீழே உள்ள 3-நிலை பதிப்பு படிக்க மட்டும் மற்றும் பக்க விளைவுகளை கலந்துள்ள முகவருக்கான மிகச்சிறிய பயனுள்ள பதிப்பாகும்.

கீழே உள்ள வகைப்பின்பகரி முக்கிய வார்த்தை சார்ந்த heuristics பயன்படுத்துகிறது ஆக Demo தீர்மானிக்கப்பட்ட மற்றும் மலிவாக இருக்கும். உற்பத்தி முறையில் இதை கற்றுக்கொண்ட வகைப்பின்பகரி அல்லது கொள்கை இயந்திரத்துடன் மாற்றுவீர்கள்.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## வடிவம் 3: கணக்கு பதிவு மற்றும் திருத்தம் மடக்கு

`print("Response approved.")` என்பது கணக்கு பதிவு அல்ல. நம்பிக்கைக்காக, ஒவ்வொரு கதவுக் கூட்டுரிமையும் பின்னர் கேட்க, மீண்டும் இயக்க, அல்லது ஒரு சம்பவ மதிப்பீட்டுடன் இணைக்கக்கூடிய கட்டமைக்கப்பட்ட நிகழ்வாக பதிவு செய்யப்பட வேண்டும்.

இரு பகுதிகள்:

1. **ஒட்டுமொத்தமாக சேர்க்கப்படும் JSONL.** ஒவ்வொரு முடிவுக்கும் ஒரு கோலம், காலச் சின்னம், செயல், நிலை, முடிவு, காரணம். தேட எளிது, பின்னர் உண்மையான கணக்கு சேமிப்பிடம் அனுப்ப எளிது.
2. **திருத்தம் மடக்கு நிராகரிப்பில்.** கதவு `deny` ஐ திருப்பிய போது, முகவர் நிராகரிக்கும் காரணத்தை கருத்தில் கொண்டு தன்னை மீண்டும் கேட்கிறது, இதனால் அடுத்த முன்மொழிவு பிரச்சினையைத் தவிர்க்கலாம்.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## கூடுதல் வளங்கள்

பல பிற பொதுவான திட்டங்கள் இந்த HITL மாதிரிகளின் வேறுபாடுகளை நடைமுறைப்படுத்துகின்றன. உங்கள் ஸ்டாக்குக்கு பொருத்தமானதை கண்டுபிடிக்க முறைகளை ஒப்பிடுங்கள்:

- **LangChain** மனிதக் கட்டுப்பாட்டிற்கு உள்ள கருவி வளைகள் ([விவரங்கள்](https://python.langchain.com/docs/integrations/tools/human_tools)): மனித உள்ளீட்டுக்காக நிறுத்தும் கருவி வளைகள்.
- **AutoGen** `UserProxyAgent` ([v0.2 விவரங்கள்](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ இதை மறுசீரமைத்தது): பன்முக முகவரிகள் உரையாடல்களில் மனிதரை பிரதிநிதித்துவப்படுத்த்வதற்கான முகவர் பாத்திரத்தைப் பயன்படுத்துகிறது.
- **Semantic Kernel** செயல்பாட்டு வடிகள் ([விவரங்கள்](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): ஒவ்வொரு செயல்பாட்டுக் கூப்பிடுதலை அணுக்கி இயங்கும் நடுவிளைவணி, நறுக்கமான தர்க்கத்திற்கு பொருத்தமானது.

ஒவ்வொரு திட்டமும் மூன்று துணை-மாதிரிகளை வெவ்வேறு விதமாக கையாள்கிறது: LangChain அவற்றை கருவிகளாக அட்டை வைப்பது, AutoGen ஒரு முகவர் பாத்திரத்தை பயன்படுத்துவது, Semantic Kernel நடுவிளைவணி வடிகளைக் கையாள்வது. உங்கள் சொந்த முகவரிக்கான வடிவமைப்பை தேர்ந்தெடுக்கும் முன் ஒரு அல்லது இரண்டு நடைமுறைகள் முழுமையாக படியுங்கள்.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
